# Build a vector store from the DispatchDesk corpus

Load every operational Markdown document in `data/corpus`, split it by its `##` sections, embed each chunk, and persist the resulting Chroma vector store locally. Run this notebook from the repository root.

Install the notebook dependencies first: `pip install langchain langchain-community langchain-chroma langchain-huggingface sentence-transformers` (or add them to the project's dependency configuration). The first embedding run downloads the selected model.

In [ ]:
from pathlib import Path
import re

from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document

def find_repo_root(start: Path) -> Path:
    """Find the repository root from the kernel working directory or notebook path."""
    for candidate in (start, *start.parents):
        if (candidate / "data" / "corpus").is_dir():
            return candidate
    raise FileNotFoundError(
        f"Could not find data/corpus from {start}. Start Jupyter inside the repository."
    )


# If the kernel starts in notebooks/, use its parent; otherwise search upward.
KERNEL_DIR = Path.cwd().resolve()
REPO_ROOT = (
    KERNEL_DIR.parent
    if KERNEL_DIR.name == "notebooks" and (KERNEL_DIR.parent / "data" / "corpus").is_dir()
    else find_repo_root(KERNEL_DIR)
)
CORPUS_DIR = REPO_ROOT / "data" / "corpus"
VECTORSTORE_DIR = REPO_ROOT / "data" / "vectorstore"
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

print(f"Corpus: {CORPUS_DIR}")
print(f"Vector store: {VECTORSTORE_DIR}")

## Load and chunk source documents

The corpus README is documentation, so it is excluded from operational retrieval. Each document is split on level-two headings to keep policy sections together. Each chunk retains the corpus `doc_id`, title, version, section, and relative source path. The first chunk also includes the document's introductory text before the first section heading.

In [ ]:
def parse_metadata(markdown: str, source: Path) -> dict[str, str]:
    title_match = re.search(r"(?m)^#\s+(.+?)\s*$", markdown)
    doc_id_match = re.search(r"(?im)^\*\*Document ID:\*\*\s*(.+?)\s*$", markdown)
    version_match = re.search(r"(?im)^\*\*Version:\*\*\s*(.+?)\s*$", markdown)
    return {
        "doc_id": doc_id_match.group(1).strip() if doc_id_match else source.stem,
        "title": title_match.group(1).strip() if title_match else source.stem,
        "version": version_match.group(1).strip() if version_match else "unknown",
        "source": source.relative_to(REPO_ROOT).as_posix(),
    }


def section_chunks(markdown: str) -> list[tuple[str, str]]:
    """Return (section heading, section text) pairs, split at ## headings."""
    matches = list(re.finditer(r"(?m)^##\s+(.+?)\s*$", markdown))
    if not matches:
        return [("Document", markdown.strip())] if markdown.strip() else []

    sections: list[tuple[str, str]] = []
    intro = markdown[:matches[0].start()].strip()
    if intro:
        sections.append(("Introduction", intro))

    for index, match in enumerate(matches):
        end = matches[index + 1].start() if index + 1 < len(matches) else len(markdown)
        heading = match.group(1).strip()
        body = markdown[match.start():end].strip()
        if body:
            sections.append((heading, body))
    return sections


source_files = sorted(path for path in CORPUS_DIR.glob("*.md") if path.name.lower() != "readme.md")
if not source_files:
    raise FileNotFoundError(f"No operational Markdown documents found in {CORPUS_DIR}")

documents: list[Document] = []
for source in source_files:
    markdown = source.read_text(encoding="utf-8")
    metadata = parse_metadata(markdown, source)
    for section, content in section_chunks(markdown):
        documents.append(Document(page_content=content, metadata={**metadata, "section": section}))

print(f"Loaded {len(source_files)} documents into {len(documents)} section chunks")
for document in documents:
    print(f"{document.metadata['doc_id']} | {document.metadata['section']}")

## Embed and persist the vector store

This writes a persistent local Chroma database under `data/vectorstore`. Re-running the build clears the existing collection first, so source edits are reflected without leaving stale chunks.

In [ ]:
embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
)

VECTORSTORE_DIR.mkdir(parents=True, exist_ok=True)
vectorstore = Chroma(
    collection_name="dispatchdesk_corpus",
    embedding_function=embeddings,
    persist_directory=str(VECTORSTORE_DIR),
)

# Rebuild deterministically from the current corpus.
existing = vectorstore.get(include=[])
if existing["ids"]:
    vectorstore.delete(ids=existing["ids"])

vectorstore.add_documents(documents)
print(f"Stored {len(documents)} embedded chunks in {VECTORSTORE_DIR}")

## Check retrieval

Use this cell to confirm that the persisted store returns relevant sections and their source metadata.

In [ ]:
query = "When should two orders be considered for batching, and what cold-chain limits apply?"
for result in vectorstore.similarity_search_with_relevance_scores(query, k=4):
    document, score = result
    print(f"score={score:.3f} | {document.metadata}")
    print(document.page_content[:700], "\n")